# booth-gpu — GPU vs CPU benchmark (Colab, free T4)

Re-expresses the booth-detection hot path (color mask + connected-components + dedup/merge) as **pure GPU tensor ops** and measures the real speedup against the production CPU code.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Then `Runtime → Run all`. The notebook clones the repo, installs deps, builds fixtures, runs the benchmark, and runs the parity check.

In [ ]:
# 0. Confirm we actually have a GPU
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU. Runtime > Change runtime type > GPU (T4), then Run all again.')
!nvidia-smi -L 2>/dev/null || true

In [ ]:
# 1. Clone the repo
import os
REPO_URL = 'https://github.com/mayanksoninls161-beep/booth-gpu.git'
if not os.path.isdir('booth-gpu'):
    !git clone $REPO_URL
%cd booth-gpu
!git pull   # make sure we have the latest pushed fixes
!ls -1 src

In [ ]:
# 2. Deps. Colab already ships a CUDA torch; we only need headless OpenCV
#    for the CPU reference (cv2.inRange / connectedComponents / intersectConvexConvex).
!pip -q install opencv-python-headless

In [ ]:
# 3. Build fixtures (synthetic dense plan: ~600 true cells -> ~2k box pool)
!python src/make_fixtures.py --n-cells 600

In [ ]:
# 4. THE BENCHMARK: CPU vs GPU, with parity, for merge + mask/components
!python src/bench.py --n-cells 600

In [ ]:
# 5. Parity check (GPU output == production CPU output within tolerance)
!python tests/test_parity.py

## Use the REAL pre-NMS pool (optional)

The synthetic pool mimics a dense plan's structure. To benchmark on the *actual* box pool your pipeline produces:

1. Inside the container, copy `src/export_real_fixtures.py` to `/app`, run it in front of one dense-plan detection (see the file's docstring), and copy the resulting `boxes_pool_real.json` into `fixtures/` here.
2. Then:

```python
!python src/bench.py --pool fixtures/boxes_pool_real.json
```

The fixture is bbox+score only — no source pixels, no secrets.

In [ ]:
# 6. (optional) bigger pool to see how the GPU gap widens with N
!python src/bench.py --n-cells 1200 --no-image

## Try it on YOUR real plan (image or PDF)

Upload a floor-plan **image (png/jpg)** or **PDF**, run the full GPU pipeline on it,
and view the annotated result inline.

**Defaults mirror the production dense preset** (`app/adaptive/config.py`): render at
**250 DPI**, **tiled** detection (`--tile 1800 --overlap 400`), pool the colour +
bordered passes (`--mode both`), merge with exact greedy NMS at **IoU 0.4 /
containment 0.6**, and a **scaled component floor** (`--min-area-frac 0.0005` of the
crop area) so small booths survive at any DPI. Just run cell 7b as-is.

Knobs if you need them:
- `--mode bordered` keys only on the grid lines (every enclosed cell, incl. pale/white);
  `--mode color` keys only on fill colour; `--mode both` (default) pools both.
- `--tile 0` forces a single full-image pass (no tiling/seam/merge).
- smaller tiles → more crops, each small booth a bigger fraction of its tile: `--tile 1000 --overlap 300`.
- `--min-area` is a hard pixel floor under the scaled `--min-area-frac`.
- `--merge cluster` is faster but collapses overlap chains into one box (under-counts).
- Big plans: add `--downscale 0.5` if it's slow.
- Nothing leaves the Colab VM.

In [ ]:
# 7a. PDF support (only needed if you upload a .pdf). Images need nothing extra.
!pip -q install PyMuPDF

In [ ]:
# 7b. Upload your plan (image or PDF), run the GPU pipeline, show the result.
from google.colab import files
from IPython.display import Image, display

# pull the latest code (JSON export + prod-aligned defaults live in src/run_real.py)
!cd /content/booth-gpu && git pull origin main

uploaded = files.upload()                 # pick one image or PDF
src_path = next(iter(uploaded))
print('uploaded:', src_path)

# Defaults already match the prod DENSE preset: dpi 250, tiled (1800/400),
# --mode both, merge iou 0.4 / containment 0.6, --min-area-frac 0.0005.
!cd /content/booth-gpu && python src/run_real.py --input "/content/$src_path"

stem = src_path.rsplit('.', 1)[0]
print('\n--- detected boxes ---'); display(Image(f'/content/booth-gpu/out/{stem}_boxes.png'))
print('--- preview mask ---');     display(Image(f'/content/booth-gpu/out/{stem}_mask.png'))

# Download the full-res JSON so the result can be verified offline.
files.download(f'/content/booth-gpu/out/{stem}_boxes.json')